# Frame Your Lane as an ML Task (ML-03)
**Intern:** Taiba Abid Jahangir  
**Track:** Machine Learning — Week 2  

This notebook maps our content health analysis onto a formal Machine Learning framework before we begin modeling.

In [14]:
import os
import pandas as pd
import numpy as np

# Create the data directory path in Colab's workspace
os.makedirs('data/raw', exist_ok=True)

# Download the starter dataset directly from the FlyRank public repo
!wget -O data/raw/content_refresh_anonymized.csv https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv

# Load the dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Verify it loaded correctly
print(f"Dataset shape: {df.shape}")

--2026-07-15 20:30:29--  https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6727670 (6.4M) [text/plain]
Saving to: ‘data/raw/content_refresh_anonymized.csv’

data/raw/content_re 100%[===================>]   6.42M  24.5MB/s    in 0.3s    

2026-07-15 20:30:30 (24.5 MB/s) - ‘data/raw/content_refresh_anonymized.csv’ saved [6727670/6727670]

Dataset shape: (30000, 44)


### 1) My Lane as an ML Task
* **Task Type:** Binary Classification.
* **The Decision:** Deciding which content items are genuinely declining and require a manual content refresh.
* **Who Acts:** Content editors and managers who will prioritize their queue based on this list.
* **The Action:** The editors manually audit and rewrite/refresh the high-risk flagged content.

### 2) Target or Proxy
* **Target Column:** `is_declining_label`
* **Type:** Observed target. It is a binary flag indicating whether a page's performance is declining.
* **Crucial Rule Guardrail:** `is_declining_label` is derived from `trend_direction`, which is computed from `trend_pct`. To prevent severe data leakage, both `trend_direction` and `trend_pct` **must be completely excluded** from the feature set.

### 3) Success Metric
Since this is an imbalanced binary classification problem where a wrong call has clear operational costs, we will use:
1. **ROC-AUC:** To evaluate the model's overall ability to distinguish between declining and stable pages across all thresholds.
2. **Precision-Recall (PR) AUC & Precision at K:**
   * **False Positives** cost wasted editor hours auditing stable pages.
   * **False Negatives** cost missed revenue/traffic on unaddressed declining pages.
   We will monitor Precision to minimize wasted editor time while maximizing Recall to catch critical declines.

In [15]:
# Create a copy to process
clean_df = df.copy()

# 1. Create the target label FIRST using the correct value 'down'
clean_df['is_declining_label'] = (clean_df['trend_direction'] == 'down').astype(int)

# 2. Fix average position (0 means "no data", not position zero)
clean_df['avg_position'] = clean_df['avg_position'].replace(0, np.nan)

# 3. Handle missingness without injecting a false category signal via blind fillna(0)
clean_df['has_word_count_missing'] = clean_df['word_count'].isna().astype(int)

# 4. Strip out the explicit leakage columns and individual content IDs
leakage_cols = ['trend_direction', 'trend_pct']
id_cols = ['content_id']

cols_to_drop = leakage_cols + id_cols
clean_df = clean_df.drop(columns=cols_to_drop, errors='ignore')

# 5. Separate features (X) and target (y)
y = clean_df['is_declining_label']
X = clean_df.drop(columns=['is_declining_label'])

# Verify shape and target class distribution
print("Processed Dataset Shape (Unit of Analysis):", clean_df.shape)
print("\nTarget Class Distribution (Base Rate):")
print(y.value_counts(normalize=True))

# Display the final unit of analysis
clean_df.head()

Processed Dataset Shape (Unit of Analysis): (30000, 43)

Target Class Distribution (Base Rate):
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,is_declining_label,has_word_count_missing
0,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,1,0
1,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,1,0
2,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,1,0
3,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,0,1
4,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,1,0


## 5) Why ML Beats a Fixed Rule Here
A plain programmatic rule (like an if-statement) is insufficient for this task because:
1. **Complex Feature Interdependencies:** Key signals like click-through rates, scroll rates, search positions, and traffic percentages do not scale linearly.
2. **Client Variance:** What constitutes an actual performance decline differs across the 32 unique clients. A static rule cannot scale dynamically across all of them.
3. **Noisy and Shifting Signals:** Our metrics are collected from different tracking environments (GA4 and Search Console) and are subject to noise, meaning ML is needed to capture the messy, shifting underlying patterns.

## 6) Self-Check
* **Is the target observed or defined by a rule?** Observed (`is_declining_label`).
* **Is the metric named before training?** Yes (ROC-AUC and Precision-Recall).
* **Are the leakage columns removed?** Yes (`trend_direction` and `trend_pct` are fully dropped).
* **Is the unit of analysis defined?** Yes (one row per content item over a 90-day window).